In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sasviya.ml.linear_model import LinearRegression

# We also set basic display and visualisation options to improve the readability of outputs and plots throughout the notebook 
%matplotlib inline

# Display all rows and columns in output if needed
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)
# train = pd.read_csv("data_preview.csv")

In [2]:

print(train.head())

NameError: name 'train' is not defined

In [ ]:
print("Data Type:\n", train.dtypes)
print(f"There are {train.shape[1]} columns and {train.shape[0]} rows in the train dataset.")
print("Column names and data type of each column:")
print(train.dtypes)
print("Summary statistics for the numeric columns:")
print(train.describe())
print("Checking for missing values in each column:")
print(train.isnull().sum())


In [ ]:
# Identify numeric and categorical columns
numeric_cols = train.select_dtypes(include=["int64", "float64"]).columns.tolist()
print("Numeric Cols:\n", numeric_cols)

categoric_cols = train.select_dtypes(include=["object"]).columns.tolist()
print("\nCategoric Cols:\n", categoric_cols)

# Create derived variable (example: GENERAL_MED)
train["GENERAL_MED"] = (train["DEPARTMENT"] == "GENERAL MED").astype(int)

# Drop PATIENT_NUMBER column if desired
# train.drop(columns="PATIENT_NUMBER", inplace=True)

# Convert categorical variables (DEPARTMENT) using one-hot encoding
train = pd.get_dummies(train, columns=["DEPARTMENT"], drop_first=True)

# Preview dataset
train.tail()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(train['ADMIT_LOS'], bins=30)
plt.title('Original Length of Stay Distribution')
plt.xlabel('Length of Stay')
plt.ylabel('Frequency')
plt.show()

In [ ]:
#train_encoded = pd.get_dummies(train, columns=categoric_cols, drop_first=True)
train_encoded = train


# Convert boolean columns to integers
bool_cols = train_encoded.select_dtypes(include="bool").columns
train_encoded[bool_cols] = train_encoded[bool_cols].astype(int)

train_encoded.shape

In [ ]:
# Create your feature set and target variable
X = train_encoded.drop("ADMIT_LOS", axis=1)
y = train_encoded["ADMIT_LOS"]

# Now perform your 3-way validation split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.3, random_state=42
)

print(f"Train features: {list(X_train.columns)}")

In [ ]:
from sasviya.ml.linear_model import Ridge, Lasso
from sasviya.ml.tree import ForestRegressor

# Define all four models in a dictionary
sas_models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=0.1),
    "Lasso": Lasso(),
    "Random Forest": ForestRegressor(n_estimators=100, max_depth=5, random_state=42)
}

# Train each model on the training set
for name, model in sas_models.items():
    model.fit(X_train, y_train)
    print(f"{name} model trained.")

# Generate and store predictions on the test set for each model
sas_predictions = {}
for name, model in sas_models.items():
    sas_predictions[name] = model.predict(X_test)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

# Define models
sk_models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=0.1),
    "Lasso": Lasso(),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
}

# Train models
for name, model in sk_models.items():
    model.fit(X_train, y_train)
    print(f"{name} model trained.")

# Generate predictions
sk_predictions = {}
for name, model in sk_models.items():
    sk_predictions[name] = model.predict(X_test)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

# Combine predictions from both frameworks for a true comparison
all_predictions = {}
for name, y_pred in sk_predictions.items():
    all_predictions[f"{name} (scikit-learn)"] = y_pred
for name, y_pred in sas_predictions.items():
    all_predictions[f"{name} (SAS Viya)"] = y_pred

# Collect and calculate metrics
results = []

for model_label, y_pred in all_predictions.items():
    r2 = r2_score(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)

    results.append(
        {
            "Model Framework": model_label,
            "R²": round(r2, 4),
            "RMSE": round(rmse, 2),
            "MAE": round(mae, 2),
        }
    )

# Convert to DataFrame and sort by best performance (highest R²)
results_df = (
    pd.DataFrame(results).sort_values("R²", ascending=False).reset_index(drop=True)
)

print("\n" + "=" * 20 + " Model Comparison " + "=" * 20)
print(results_df.to_string(index=False))
print("=" * 59 + "\n")

# 3. Generate diagnostic plots for each model
for model_label, y_pred in all_predictions.items():
    r2 = r2_score(y_test, y_pred)
    residuals = y_test - y_pred

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    fig.suptitle(model_label, fontsize=14, fontweight="bold")

    # Plot 1: Actual vs Predicted
    ax1 = axes[0]
    ax1.scatter(
        y_test,
        y_pred,
        alpha=0.4,
        color="purple",
        edgecolors="white",
        linewidths=0.3,
    )
    ax1.plot(
        [y_test.min(), y_test.max()],
        [y_test.min(), y_test.max()],
        "--",
        color="red",
        linewidth=1.2,
        label="Perfect Fit",
    )
    ax1.text(
        0.05,
        0.90,
        f"R² = {r2:.4f}",
        transform=ax1.transAxes,
        fontsize=11,
        color="darkblue",
        fontweight="bold",
    )
    ax1.set_xlabel("Actual Length of Stay")
    ax1.set_ylabel("Predicted Length of Stay")
    ax1.set_title("Actual vs. Predicted")
    ax1.legend()
    ax1.grid(True, linestyle=":", alpha=0.6)

    # Plot 2: Residual Plot
    ax2 = axes[1]
    ax2.scatter(
        y_pred,
        residuals,
        alpha=0.4,
        color="teal",
        edgecolors="white",
        linewidths=0.3,
    )
    ax2.axhline(0, color="red", linestyle="--", linewidth=1.2)
    ax2.set_xlabel("Predicted Length of Stay")
    ax2.set_ylabel("Residuals (Actual − Predicted)")
    ax2.set_title("Residual Plot")
    ax2.grid(True, linestyle=":", alpha=0.6)

    plt.tight_layout()
    plt.show()


    # Load the separate submission test dataset
test_df = pd.read_csv("data_preview.csv")

X_submission = test_df.copy()

# Add missing columns that X_train has, filled with 0
missing_cols = set(X_train.columns) - set(X_submission.columns)
for col in missing_cols:
    X_submission[col] = 0

X_submission = X_submission[X_train.columns]

# Auto-select the best model name and metrics from the results DataFrame
best_model_label = results_df.iloc[0]["Model Framework"]
best_r2 = results_df.iloc[0]["R²"]
best_rmse = results_df.iloc[0]["RMSE"]

print(
    f"Best Model Selected: {best_model_label} (Local R² = {best_r2}, Local RMSE = {best_rmse})"
)

# Extract the correct model object by checking the framework suffix
if "(scikit-learn)" in best_model_label:
    pure_name = best_model_label.replace(" (scikit-learn)", "")
    best_model = sk_models[pure_name]
elif "(SAS Viya)" in best_model_label:
    pure_name = best_model_label.replace(" (SAS Viya)", "")
    best_model = sas_models[pure_name]
# --- TEST CHECK ---
# Run a quick sanity check against your isolated X_test to ensure no overfitting
from sklearn.metrics import mean_squared_error, r2_score
test_preds = best_model.predict(X_test)
test_r2 = r2_score(y_test, test_preds)
test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))

print(f"Holdout Local Test Check:")
print(f"   + Test R²: {test_r2:.4f}")
print(f"   + Test RMSE: {test_rmse:.4f}\n")
# ---------------------------------------

# Generate predictions on the test dataset
print(f"Generating final predictions on 'data_preview.csv'...")
raw_predictions = best_model.predict(X_submission)
submission_predictions = np.clip(np.round(raw_predictions), 1, None).astype(int)

# Create and save the submission file
submission_df = pd.DataFrame(
    {"ENCOUNTER_KEY": test_df["ENCOUNTER_KEY"].values, "ADMIT_LOS": submission_predictions}
)

submission_df.to_csv("submission.csv", index=False)
print(
    f"\nSubmission file created successfully: 'submission.csv' ({len(submission_df)} rows)"
)